# Chapter 10 — The Inverse and Implicit Function Theorems

Companion notebook to Chapter 10 of *Advanced Calculus for Machine Learning*.
Every numerical claim of the chapter is recomputed here with PyTorch and
checked with an assertion.

In [1]:
import math
import warnings
import torch
from torch.func import jacrev, hessian, vjp

warnings.filterwarnings("ignore", category=FutureWarning)
torch.set_default_dtype(torch.float64)
torch.manual_seed(0)
op = lambda A: torch.linalg.matrix_norm(A, ord=2).item()
print(torch.__version__)

2.15.0+cu132


## 1. The inverse function theorem

Figure 10.1 and Exercise 10.3: the iteration of the proof,
`x <- x + Df(a)^{-1} (y - f(x))`, with the differential frozen at `a`.
Then Theorem 10.2 on a map of the plane: the local inverse computed by this
iteration, its Lipschitz bound `2 |A^{-1}|`, and
`D(f^{-1})(y) = Df(f^{-1}(y))^{-1}`.

In [2]:
# Figure 10.1: f(x) = x^2, a = 1, y = 2.2
x, xs = 1.0, [1.0]
for _ in range(40):
    x = x + (2.2 - x * x) / 2.0; xs.append(x)
print("iterates:", [round(v, 4) for v in xs[:5]], " limit", round(xs[-1], 6), " sqrt(2.2) =", round(math.sqrt(2.2), 6))
assert [round(v, 4) for v in xs[:4]] == [1.0, 1.6, 1.42, 1.5118] and abs(xs[-1] - math.sqrt(2.2)) < 1e-10
ratios = [abs(xs[k + 1] - math.sqrt(2.2)) / abs(xs[k] - math.sqrt(2.2)) for k in range(10, 20)]
print("linear rate:", round(ratios[-1], 4), " = |1 - f'(x*)/f'(a)| =", round(abs(1 - math.sqrt(2.2)), 4))

# Exercise 10.3
f = lambda p: torch.stack([p[0] + p[1] ** 2, p[1] + p[0] ** 2])
y = torch.tensor([0.1, 0.2])
x = torch.zeros(2); its = []
for _ in range(60):
    x = x + (y - f(x)); its.append(x.clone())
print("x1, x2, x3 =", [[round(v, 4) for v in p.tolist()] for p in its[:3]], "  solution", [round(v, 4) for v in x.tolist()])
assert torch.allclose(its[0], torch.tensor([0.1, 0.2])) and torch.allclose(its[1], torch.tensor([0.06, 0.19]))
assert torch.allclose(its[2], torch.tensor([0.0639, 0.1964]), atol=5e-5)
assert torch.allclose(f(x), y, atol=1e-14) and torch.allclose(x, torch.tensor([0.0615, 0.1962]), atol=5e-5)

iterates: [1.0, 1.6, 1.42, 1.5118, 1.469]  limit 1.48324  sqrt(2.2) = 1.48324
linear rate: 0.4832  = |1 - f'(x*)/f'(a)| = 0.4832
x1, x2, x3 = [[0.1, 0.2], [0.06, 0.19], [0.0639, 0.1964]]   solution [0.0615, 0.1962]


In [3]:
torch.manual_seed(1)
B1, B2 = torch.randn(2, 2), torch.randn(2, 2)
f = lambda p: p + 0.3 * torch.tanh(B1 @ p) + 0.2 * torch.sin(B2 @ p)       # a map of the plane
a = torch.tensor([0.4, -0.2]); b = f(a); A = jacrev(f)(a); Ainv = torch.linalg.inv(A)

def local_inverse(y, steps=200):
    x = a.clone()
    for _ in range(steps):
        x = x + Ainv @ (y - f(x))
    return x

ys = [b + 0.1 * torch.randn(2) for _ in range(40)]
xs = [local_inverse(y) for y in ys]
assert all(torch.allclose(f(x), y, atol=1e-12) for x, y in zip(xs, ys))
lip = max(((xs[i] - xs[j]).norm() / (ys[i] - ys[j]).norm()).item() for i in range(40) for j in range(i))
print("largest |phi(y1) - phi(y2)| / |y1 - y2| =", round(lip, 4), " <= 2 |A^{-1}| =", round(2 * op(Ainv), 4))
assert lip <= 2 * op(Ainv)

y0 = ys[0]; x0 = xs[0]
Dphi = jacrev(lambda y: local_inverse(y, 300))(y0)       # differentiate through the iterations
print("D(f^-1)(y0) by differentiating the iteration:\n", Dphi)
print("Df(f^-1(y0))^-1:\n", torch.linalg.inv(jacrev(f)(x0)))
assert torch.allclose(Dphi, torch.linalg.inv(jacrev(f)(x0)), atol=1e-9)

largest |phi(y1) - phi(y2)| / |y1 - y2| = 0.9415  <= 2 |A^{-1}| = 1.8749
D(f^-1)(y0) by differentiating the iteration:
 tensor([[ 0.8980, -0.0328],
        [ 0.1715,  0.7958]])
Df(f^-1(y0))^-1:
 tensor([[ 0.8980, -0.0328],
        [ 0.1715,  0.7958]])


### The theorem is local, and continuity of the differential is needed

Exercise 10.1: `(e^x cos y, e^x sin y)` has an invertible differential
everywhere and is not injective. Exercise 10.2: `x/2 + x^2 sin(1/x)` has
`f'(0) = 1/2` and is not injective near 0.

In [4]:
fe = lambda p: torch.stack([torch.exp(p[0]) * torch.cos(p[1]), torch.exp(p[0]) * torch.sin(p[1])])
for _ in range(100):
    p = 2 * torch.randn(2)
    assert abs(torch.linalg.det(jacrev(fe)(p)).item() - math.exp(2 * p[0].item())) < 1e-9 * math.exp(2 * p[0].item())
p = torch.tensor([0.3, 0.5])
assert torch.allclose(fe(p), fe(p + torch.tensor([0.0, 2 * math.pi])))
inv = lambda q: torch.stack([0.5 * torch.log(q[0] ** 2 + q[1] ** 2), torch.atan2(q[1], q[0])])
assert torch.allclose(inv(fe(p)), p)
print("D(f^-1)(1, 0) =", jacrev(inv)(torch.tensor([1.0, 0.0])).tolist())
assert torch.allclose(jacrev(inv)(torch.tensor([1.0, 0.0])), torch.eye(2))

g = lambda x: x / 2 + x * x * math.sin(1 / x)
dg = lambda x: 0.5 + 2 * x * math.sin(1 / x) - math.cos(1 / x)
for k in (1, 10, 100, 1000):
    xk, yk = 1 / (2 * math.pi * k), 1 / ((2 * k + 1) * math.pi)
    assert abs(dg(xk) + 0.5) < 1e-9 and abs(dg(yk) - 1.5) < 1e-9
print("f'(1/(2 pi k)) = -1/2 and f'(1/((2k+1) pi)) = 3/2 for all k checked; f'(0) = 1/2")
# two different points with the same value, arbitrarily close to 0: around a local maximum
k = 50; lo, hi = 1 / ((2 * k + 1) * math.pi), 1 / (2 * math.pi * k)       # f' goes from 3/2 to -1/2
for _ in range(80):                                                         # locate the maximum
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if dg(mid) > 0 else (lo, mid)
xm = lo; left, right = xm - 2e-6, xm + 2e-6
print("local maximum at", xm, ": f(left) < f(max) > f(right):", g(left) < g(xm) > g(right))
assert g(left) < g(xm) and g(right) < g(xm)

D(f^-1)(1, 0) = [[1.0, 0.0], [0.0, 1.0]]
f'(1/(2 pi k)) = -1/2 and f'(1/((2k+1) pi)) = 3/2 for all k checked; f'(0) = 1/2
local maximum at 0.0031725875296096826 : f(left) < f(max) > f(right): True


## 2. The implicit function theorem

Example 10.6 (circle), Exercises 10.4 and 10.5: the differential of the
implicit function from `D phi = -D_2 G^{-1} D_1 G`, compared with a solution
computed by Newton's method in `y` and differentiated numerically.

In [5]:
def implicit_derivative(G, x, y):
    Jx = jacrev(G, argnums=0)(x, y); Jy = jacrev(G, argnums=1)(x, y)
    return -torch.linalg.solve(Jy.reshape(y.numel(), y.numel()), Jx.reshape(y.numel(), -1))

def solve_y(G, x, y0, steps=60):                   # Newton's method in y, for fixed x
    y = y0.clone()
    for _ in range(steps):
        Jy = jacrev(G, argnums=1)(x, y).reshape(y.numel(), y.numel())
        y = y - torch.linalg.solve(Jy, G(x, y).reshape(-1))
    return y

def numeric(G, x, y0, e=1e-6):
    return (solve_y(G, x + e, y0) - solve_y(G, x - e, y0)) / (2 * e)

circle = lambda x, y: (x ** 2 + y ** 2 - 1).reshape(1)
x, y = torch.tensor([0.6]), torch.tensor([0.8])
d = implicit_derivative(circle, x, y)
print("circle at (0.6, 0.8): phi' =", d.item(), "  numeric:", numeric(circle, x, y).item())
assert abs(d.item() + 0.75) < 1e-12 and abs(numeric(circle, x, y).item() + 0.75) < 1e-7
print("at (1, 0): dG/dy =", jacrev(circle, argnums=1)(torch.tensor([1.0]), torch.tensor([0.0])).item(),
      "  dG/dx =", jacrev(circle, argnums=0)(torch.tensor([1.0]), torch.tensor([0.0])).item())

folium = lambda x, y: (x ** 3 + y ** 3 - 3 * x * y).reshape(1)
x, y = torch.tensor([1.5]), torch.tensor([1.5])
assert abs(folium(x, y).item()) < 1e-12
print("folium at (3/2, 3/2): phi' =", implicit_derivative(folium, x, y).item())
assert abs(implicit_derivative(folium, x, y).item() + 1) < 1e-12
z = torch.tensor([0.0])
assert jacrev(folium, argnums=0)(z, z).item() == 0 and jacrev(folium, argnums=1)(z, z).item() == 0

system = lambda x, yz: torch.stack([x[0] ** 2 + yz[0] ** 2 + yz[1] ** 2 - 3, x[0] + 2 * yz[0] + 3 * yz[1] - 6])
x, yz = torch.tensor([1.0]), torch.tensor([1.0, 1.0])
d = implicit_derivative(system, x, yz)
print("sphere and plane at (1,1,1): (y', z') =", d.flatten().tolist(), "  numeric:",
      [round(v, 6) for v in numeric(system, x, yz).tolist()])
assert torch.allclose(d.flatten(), torch.tensor([-2.0, 1.0]))
assert torch.allclose(numeric(system, x, yz), torch.tensor([-2.0, 1.0]), atol=1e-6)
tangent = lambda p: torch.stack([p[0] ** 2 + p[1] ** 2 + p[2] ** 2 - 3, p[0] + p[1] + p[2] - 3])
print("with the tangent plane, rank of the Jacobian:",
      torch.linalg.matrix_rank(jacrev(tangent)(torch.ones(3))).item())
assert torch.linalg.matrix_rank(jacrev(tangent)(torch.ones(3))).item() == 1

circle at (0.6, 0.8): phi' = -0.75   numeric: -0.7499999999938112


at (1, 0): dG/dy = 0.0   dG/dx = 2.0
folium at (3/2, 3/2): phi' = -1.0
sphere and plane at (1,1,1): (y', z') = [-2.0, 1.0]   numeric: [-2.0, 1.0]


with the tangent plane, rank of the Jacobian: 1


## 3. Submersions and immersions

Theorem 10.8: `Theta(x) = (f(x), pi(x - a))` is a local diffeomorphism when
`Df(a)` is surjective, and it sends the level set of `f` into a slice
`{f(a)} x K`. Theorem 10.9: `F(x, c) = f(x) + c` is a local diffeomorphism
when `Df(a)` is injective. Exercise 10.8.

In [6]:
f = lambda p: (p[0] ** 2 + p[1] ** 2 - p[2] ** 2).reshape(1)          # R^3 -> R
a = torch.tensor([1.0, 0.5, 0.8])
row = jacrev(f)(a).flatten()
_, _, Vh = torch.linalg.svd(row.reshape(1, 3)); Kbasis = Vh[1:].T          # basis of K = ker Df(a)
proj = lambda v: Kbasis.T @ v                                          # coordinates in K of pi(v)
Theta = lambda p: torch.cat([f(p), proj(p - a)])
assert abs(torch.linalg.det(jacrev(Theta)(a)).item()) > 1e-6
# a point of the level set near a, found by Newton along the gradient direction
p = a + 0.05 * Kbasis @ torch.tensor([1.0, -0.7])
for _ in range(30):
    p = p - (f(p) - f(a)) * jacrev(f)(p).flatten() / (jacrev(f)(p).flatten() @ jacrev(f)(p).flatten())
print("f(p) - f(a) =", (f(p) - f(a)).item(), "  Theta(p) =", [round(v, 6) for v in Theta(p).tolist()])
assert abs((Theta(p)[0] - f(a)).item()) < 1e-12                        # first component is f(a)
print("Df at the origin:", jacrev(f)(torch.zeros(3)).flatten().tolist(), "(not a submersion there)")

cusp = lambda t: torch.stack([t ** 2, t ** 3])
print("velocity of (t^2, t^3) at t = 0:", jacrev(cusp)(torch.tensor(0.0)).tolist(),
      "  at t = 1:", jacrev(cusp)(torch.tensor(1.0)).tolist())
assert jacrev(cusp)(torch.tensor(0.0)).norm() == 0

# immersion: a curve in R^3 and a complement C of its velocity
curve = lambda t: torch.stack([torch.cos(t[0]), torch.sin(t[0]), t[0] ** 2])
t0 = torch.tensor([0.3]); vel = jacrev(curve)(t0).flatten()
_, _, Vh = torch.linalg.svd(vel.reshape(1, 3)); Cbasis = Vh[1:].T
Fmap = lambda q: curve(q[:1]) + Cbasis @ q[1:]
assert abs(torch.linalg.det(jacrev(Fmap)(torch.cat([t0, torch.zeros(2)]))).item()) > 1e-6
print("DF(a, 0) is invertible: F(x, c) = f(x) + c is a local diffeomorphism")

f(p) - f(a) = 1.1102230246251565e-16   Theta(p) = [0.61, 0.049958, -0.035]
Df at the origin: [0.0, 0.0, 0.0] (not a submersion there)
velocity of (t^2, t^3) at t = 0: [0.0, 0.0]   at t = 1: [2.0, 3.0]
DF(a, 0) is invertible: F(x, c) = f(x) + c is a local diffeomorphism


## 4. Implicit differentiation

The fixed-point layer `z* = tanh(W z* + U theta)` (Exercise 10.7 is the
scalar case). The derivative of a loss with respect to `theta` is computed
in three ways: by backpropagating through 200 iterations of the solver, by
the forward formula `(I - D_z g)^{-1} D_theta g`, and by the reverse recipe
of the chapter (one linear system with the transposed matrix).

In [7]:
torch.manual_seed(2)
nz, nt = 6, 4
W = torch.randn(nz, nz); W = 0.7 * W / op(W); U = torch.randn(nz, nt); c = torch.randn(nz)
g = lambda theta, z: torch.tanh(W @ z + U @ theta)
loss = lambda z: 0.5 * ((z - c) ** 2).sum()

def solve(theta, steps=200):
    z = torch.zeros(nz)
    for _ in range(steps):
        z = g(theta, z)
    return z

theta = torch.randn(nt)
zstar = solve(theta)
assert (g(theta, zstar) - zstar).norm() < 1e-13

# (a) backpropagation through the 200 iterations
unrolled = jacrev(lambda th: loss(solve(th)))(theta)

# (b) forward formula: Dz* = (I - D_z g)^{-1} D_theta g
Dzg = jacrev(g, argnums=1)(theta, zstar); Dtg = jacrev(g, argnums=0)(theta, zstar)
Dzstar = torch.linalg.solve(torch.eye(nz) - Dzg, Dtg)
forward = jacrev(loss)(zstar) @ Dzstar

# (c) reverse recipe: G = z - g(theta, z); solve (D_z G)^T lam = d loss; return -(D_theta G)^T lam
lam = torch.linalg.solve((torch.eye(nz) - Dzg).T, jacrev(loss)(zstar))
reverse = Dtg.T @ lam                         # -(D_theta G)^T lam with D_theta G = -D_theta g

print("unrolled :", [round(v, 8) for v in unrolled.tolist()])
print("forward  :", [round(v, 8) for v in forward.tolist()])
print("reverse  :", [round(v, 8) for v in reverse.tolist()])
assert torch.allclose(unrolled, forward, atol=1e-10) and torch.allclose(forward, reverse, atol=1e-12)
print("|D_z g| =", round(op(Dzg), 4), "< 1, so I - D_z g is invertible")
assert op(Dzg) < 1

# scalar case, Exercise 10.7
def fixed_point(w, x, steps=300):
    z = 0.0
    for _ in range(steps):
        z = math.tanh(w * z + x)
    return z

w, x = 0.6, 0.4
z = fixed_point(w, x); u = 1 - z * z
dz_dw = (fixed_point(w + 1e-6, x) - fixed_point(w - 1e-6, x)) / 2e-6
dz_dx = (fixed_point(w, x + 1e-6) - fixed_point(w, x - 1e-6)) / 2e-6
print("scalar: dz*/dw =", z * u / (1 - w * u), " numeric:", dz_dw)
print("        dz*/dx =", u / (1 - w * u), " numeric:", dz_dx)
assert abs(z * u / (1 - w * u) - dz_dw) < 1e-8 and abs(u / (1 - w * u) - dz_dx) < 1e-8

unrolled : [0.35103245, 0.71591564, 0.40231401, 0.863094]
forward  : [0.35103245, 0.71591564, 0.40231401, 0.863094]
reverse  : [0.35103245, 0.71591564, 0.40231401, 0.863094]
|D_z g| = 0.4058 < 1, so I - D_z g is invertible
scalar: dz*/dw = 0.5601430293616696  numeric: 0.5601430294044363
        dz*/dx = 0.8452358334928042  numeric: 0.8452358334909427


### Gradient with respect to a hyperparameter

Exercise 10.6 (ridge regression with `sum x^2 = 4`, `sum x y = 6`), then a
logistic regression where `w*(lambda)` has no closed form: the derivative of
the validation loss with respect to `lambda` from one linear system with the
Hessian, compared with a finite difference of the re-trained model.

In [8]:
wstar = lambda lam: 6 / (4 + lam)
lam = 2.0
print("ridge: w* =", wstar(lam), "  dw*/dlambda =", -wstar(lam) / (4 + lam),
      "  numeric:", (wstar(lam + 1e-6) - wstar(lam - 1e-6)) / 2e-6)
assert abs(-wstar(lam) / (4 + lam) + 1 / 6) < 1e-12
dval = (wstar(lam) - 2) * (-wstar(lam) / (4 + lam))
print("derivative of the validation loss:", dval, "> 0: decrease lambda")
assert abs(dval - 1 / 6) < 1e-12

torch.manual_seed(3)
Xtr, Xva = torch.randn(60, 5), torch.randn(40, 5)
wtrue = torch.randn(5)
ytr = (torch.sigmoid(Xtr @ wtrue) > torch.rand(60)).double()
yva = (torch.sigmoid(Xva @ wtrue) > torch.rand(40)).double()
bce = lambda X, y, w: torch.nn.functional.binary_cross_entropy_with_logits(X @ w, y)
train = lambda lam, w: bce(Xtr, ytr, w) + 0.5 * lam * (w ** 2).sum()
val = lambda w: bce(Xva, yva, w)

def fit(lam):                                   # Newton's method on the training loss
    w = torch.zeros(5)
    for _ in range(50):
        w = w - torch.linalg.solve(hessian(lambda v: train(lam, v))(w), jacrev(lambda v: train(lam, v))(w))
    return w

lam = torch.tensor(0.1)
w = fit(lam)
Hww = hessian(lambda v: train(lam, v))(w)                               # D_w G
Dlam = jacrev(lambda l: jacrev(lambda v: train(l, v))(w))(lam)          # D_lambda G (= w here)
mu = torch.linalg.solve(Hww.T, jacrev(val)(w))                          # one linear system
hyper = -(Dlam @ mu)
numeric = (val(fit(lam + 1e-5)) - val(fit(lam - 1e-5))) / 2e-5
print("d(validation loss)/d(lambda): implicit", hyper.item(), "  finite difference", numeric.item())
assert torch.allclose(Dlam, w) and abs(hyper.item() - numeric.item()) < 1e-7

ridge: w* = 1.0   dw*/dlambda = -0.16666666666666666   numeric: -0.166666666689963
derivative of the validation loss: 0.16666666666666666 > 0: decrease lambda


d(validation loss)/d(lambda): implicit 0.3061809257493978   finite difference 0.3061809252691283


## 5. Invertible layers

Exercise 10.9. A coupling layer is inverted in closed form and its Jacobian
determinant is `exp(sum s)`. A residual layer `x + F(x)` with a Lipschitz
constant below 1 is inverted by the iteration `x <- y - F(x)`.

In [9]:
torch.manual_seed(4)
S1, T1 = torch.randn(3, 2), torch.randn(3, 2)
s = lambda x1: torch.tanh(S1 @ x1); t = lambda x1: torch.sin(T1 @ x1)
def coupling(x):
    x1, x2 = x[:2], x[2:]
    return torch.cat([x1, x2 * torch.exp(s(x1)) + t(x1)])
def coupling_inv(y):
    y1, y2 = y[:2], y[2:]
    return torch.cat([y1, (y2 - t(y1)) * torch.exp(-s(y1))])
x = torch.randn(5)
assert torch.allclose(coupling_inv(coupling(x)), x)
Jc = jacrev(coupling)(x)
print("log|det J| =", torch.linalg.slogdet(Jc)[1].item(), "  sum s(x1) =", s(x[:2]).sum().item())
assert abs(torch.linalg.slogdet(Jc)[1].item() - s(x[:2]).sum().item()) < 1e-12
assert torch.allclose(Jc[:2, 2:], torch.zeros(2, 3))                    # block triangular

Wr = torch.randn(5, 5); Wr = 0.8 * Wr / op(Wr)
Fres = lambda x: torch.tanh(Wr @ x)                                     # Lipschitz constant <= 0.8
res = lambda x: x + Fres(x)
y = res(x); xk = torch.zeros(5); errs = []
for _ in range(120):
    xk = y - Fres(xk); errs.append((xk - x).norm().item())
print("inverse by iteration: error after 10, 50, 120 steps:", [f"{errs[k]:.2e}" for k in (9, 49, 119)])
assert errs[-1] < 1e-10 and all(errs[k + 1] <= 0.8 * errs[k] + 1e-15 for k in range(60))
for _ in range(200):
    p, q = 3 * torch.randn(5), 3 * torch.randn(5)
    assert (res(p) - res(q)).norm() >= (1 - 0.8) * (p - q).norm() - 1e-12
print("|f(x) - f(x')| >= (1 - kappa)|x - x'| on 200 random pairs")

log|det J| = -0.1133164175195498   sum s(x1) = -0.11331641751954981
inverse by iteration: error after 10, 50, 120 steps: ['5.16e-05', '1.28e-16', '1.28e-16']
|f(x) - f(x')| >= (1 - kappa)|x - x'| on 200 random pairs


## Summary

* The local inverse is the fixed point of `x -> x + Df(a)^{-1}(y - f(x))`;
  its differential is the inverse of the differential.
* The differential of an implicit function comes from one linear system.
* Differentiating a loss through a fixed point or through a trained model
  needs one linear system with the transposed matrix, and gives the same
  result as backpropagating through the solver.
* Coupling and contractive residual layers are invertible on the whole
  space by construction.